# ML-1M Sparse Walker: fast long-sequence loop

This notebook checks out the long-sequence speed branch and resumes the existing canonical ML-1M Walker checkpoint. Evaluation protocol is unchanged.

Speed changes: no unused state-history stacks for FullCE, batched graph-touch bookkeeping, length-bucketed batches, BF16 on supported CUDA GPUs, non-blocking transfers, and `--walker-only` to skip frozen baseline re-evaluation.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
!rm -rf /content/Sparsewalker
!git clone -q --branch agent/walker-speed-long-sequences https://github.com/hanialshater/Sparsewalker-.git /content/Sparsewalker
%cd /content/Sparsewalker
!pip -q install -e .
import torch
print('torch', torch.__version__)
print('GPU', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
print('BF16', torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False)

## Resume the current canonical ML-1M Walker run

Uses the same Drive output directory as the previous canonical-pair notebook. `last.pt` is loaded automatically when its protocol fingerprint matches. The four frozen baselines are skipped.

In [ ]:
OUT='/content/drive/MyDrive/sparsewalker_canonical_pair'
BASE='/content/drive/MyDrive/sparsewalker_esasrec_2x2'
!python benchmarks/run_canonical_pair.py \
  --dataset ml1m --seed 42 \
  --baseline-root "$BASE" --output-dir "$OUT" \
  --eval-batch-size 1024 \
  --walker-only

### What to watch

Each epoch prints `seconds`, `positions_per_s`, `padding_efficiency`, `bf16`, and `bucketed`. Send that first `TRAIN` line back and we can quantify the speedup immediately.